# Seasonal Best Model Evaluation

This notebook loads the best available model for each stored seasonal month-group, evaluates the stitched seasonal strategy over a shared evaluation window, plots predictions, and compares aggregate metrics against a single global model baseline when available.

In [5]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
from IPython.display import display
from sklearn.metrics import r2_score

from darts import TimeSeries, concatenate
from darts.dataprocessing.transformers import MissingValuesFiller, Scaler
from darts.models import NBEATSModel

warnings.filterwarnings("ignore")

BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR.parent / "cleaned_demand_data.csv"
REGISTRY_PATH = BASE_DIR / "models" / "model_registry.json"
MODELS_DIR = BASE_DIR / "models"
DARTS_LOGS_DIR = BASE_DIR / "darts_logs"
RESULTS_DIR = BASE_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

ZONE = "Total Demand"
TEST_START_DATE = "2025-01-01"
TEST_END_DATE = "2026-01-14"
EVAL_MONTHS = None
OUTPUT_CHUNK_LENGTH = 96
STRIDE = 96
MAX_PLOT_POINTS = 5000
AUTO_DISCOVER_MONTH_GROUPS = True

print(f"Notebook directory: {BASE_DIR}")
print(f"Zone: {ZONE}")
print(f"Evaluation window: {TEST_START_DATE} to {TEST_END_DATE}")

Notebook directory: c:\Users\hp\Desktop\forecasting work\Nbeats refactor\Total
Zone: Total Demand
Evaluation window: 2025-01-01 to 2026-01-14


In [6]:
def load_registry(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def load_data(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["Timestamp"] = pd.to_datetime(df["Timestamp"])
    if "Total Demand (as recorded)" in df.columns and "Total Demand" not in df.columns:
        df = df.rename(columns={"Total Demand (as recorded)": "Total Demand"})
    return df.sort_values("Timestamp").reset_index(drop=True)


def normalize_month_group(value):
    if value is None:
        return None
    if isinstance(value, str):
        if not value.strip():
            return None
        value = [int(x) for x in value.split("|") if str(x).strip()]
    return tuple(sorted(int(x) for x in value))


def month_group_label(months) -> str:
    months = normalize_month_group(months)
    if not months:
        return "all_months"
    return "months_" + "_".join(str(m) for m in months)


def extract_requested_months(meta: dict):
    for key in ("requested_months",):
        value = meta.get(key)
        if isinstance(value, list) and value:
            return value
    for section in ("training_data", "config"):
        value = meta.get(section, {}).get("requested_months") if isinstance(meta.get(section), dict) else None
        if isinstance(value, list) and value:
            return value
    return None


def filter_by_zone_and_months(df: pd.DataFrame, zone: str, start_date: str, end_date: str, months=None) -> pd.DataFrame:
    df_filtered = df[(df["Timestamp"] >= start_date) & (df["Timestamp"] <= end_date)].copy()
    if months:
        df_filtered = df_filtered[df_filtered["Timestamp"].dt.month.isin(list(months))]
    return df_filtered[["Timestamp", zone]].copy()


def prepare_test_series(df: pd.DataFrame, target_col: str, scaler: Scaler) -> TimeSeries:
    series = TimeSeries.from_dataframe(df, time_col="Timestamp", value_cols=[target_col], freq="15min")
    filler = MissingValuesFiller()
    series = filler.transform(series)
    series = scaler.transform(series)
    return series


def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, name: str = "Model") -> dict:
    errors = y_pred - y_true
    mae = float(np.mean(np.abs(errors)))
    mse = float(np.mean(errors ** 2))
    rmse = float(np.sqrt(mse))
    mape = float(np.mean(np.abs(errors / (np.abs(y_true) + 1e-8))) * 100)
    mae_pct = float((mae / (np.abs(y_true).mean() + 1e-8)) * 100)
    r2 = float(r2_score(y_true, y_pred))
    return {
        "Model": name,
        "MAE": mae,
        "RMSE": rmse,
        "MAPE": mape,
        "MAE%": mae_pct,
        "R2": r2,
        "TotalAbsError": float(np.abs(errors).sum()),
        "Bias": float(errors.mean()),
        "n_points": int(len(y_true)),
    }


def safe_load_checkpoint(model_name: str):
    torch.serialization.add_safe_globals([torch.optim.Adam])
    try:
        return NBEATSModel.load_from_checkpoint(model_name=model_name, best=True, work_dir=str(DARTS_LOGS_DIR))
    except TypeError:
        return NBEATSModel.load_from_checkpoint(model_name=model_name, best=True, work_dir=str(DARTS_LOGS_DIR))


def load_model_and_scaler(model_name: str):
    model = safe_load_checkpoint(model_name)
    scaler_path = MODELS_DIR / model_name / "scaler.joblib"
    if not scaler_path.exists():
        raise FileNotFoundError(f"Scaler not found: {scaler_path}")
    scaler = joblib.load(scaler_path)
    return model, scaler


def prediction_df_for_model(model_name: str, zone: str, start_date: str, end_date: str, months=None) -> pd.DataFrame:
    model, scaler = load_model_and_scaler(model_name)
    df_test = filter_by_zone_and_months(df_full, zone, start_date, end_date, months)
    if df_test.empty:
        raise ValueError(f"No rows after filtering for {model_name}")

    test_series_scaled = prepare_test_series(df_test, zone, scaler)
    preds_scaled = model.historical_forecasts(
        test_series_scaled,
        forecast_horizon=OUTPUT_CHUNK_LENGTH,
        stride=STRIDE,
        last_points_only=False,
        retrain=False,
        verbose=False,
    )
    if not preds_scaled:
        raise ValueError(f"No forecasts generated for {model_name}")

    preds_scaled = concatenate(preds_scaled)
    preds_unscaled = scaler.inverse_transform(preds_scaled)
    preds_df = preds_unscaled.to_dataframe().reset_index()
    if preds_df.shape[1] < 2:
        raise ValueError(f"Unexpected prediction shape for {model_name}")
    preds_df = preds_df.rename(columns={preds_df.columns[0]: "Timestamp", preds_df.columns[1]: "Predicted"})[["Timestamp", "Predicted"]]
    preds_df["Timestamp"] = pd.to_datetime(preds_df["Timestamp"])

    actual_df = df_test.rename(columns={zone: "Actual"})
    actual_df["Timestamp"] = pd.to_datetime(actual_df["Timestamp"])
    merged = pd.merge(actual_df, preds_df, on="Timestamp", how="inner").dropna().sort_values("Timestamp")
    if merged.empty:
        raise ValueError(f"No overlapping timestamps for {model_name}")
    return merged


def build_selection_table(registry: dict, zone: str) -> pd.DataFrame:
    rows = []
    for model_name, meta in registry.items():
        if meta.get("zone") != zone:
            continue
        metrics = meta.get("metrics", {}) if isinstance(meta.get("metrics"), dict) else {}
        test_mape = metrics.get("test_MAPE")
        if test_mape is None or pd.isna(test_mape):
            continue
        month_group = normalize_month_group(extract_requested_months(meta))
        rows.append(
            {
                "model_name": model_name,
                "month_group": month_group,
                "month_group_label": month_group_label(month_group),
                "test_MAPE": float(test_mape),
                "test_MAE": float(metrics.get("test_MAE", np.nan)),
                "test_R2": float(metrics.get("test_R2", np.nan)),
                "created_at": meta.get("created_at"),
            }
        )
    df = pd.DataFrame(rows)
    if df.empty:
        raise ValueError(f"No evaluated models found in registry for zone {zone}")
    return df.sort_values(["month_group_label", "test_MAPE", "created_at"]).reset_index(drop=True)


def pick_best_models_per_group(selection_df: pd.DataFrame) -> pd.DataFrame:
    seasonal_df = selection_df[selection_df["month_group"].notna()].copy()
    if seasonal_df.empty:
        raise ValueError("No seasonal month-group models with test metrics were found.")
    best_df = seasonal_df.groupby("month_group_label", as_index=False).first()
    return best_df.sort_values("month_group_label").reset_index(drop=True)


def pick_best_global_model(selection_df: pd.DataFrame) -> pd.Series | None:
    global_df = selection_df[selection_df["month_group"].isna()].copy()
    if global_df.empty:
        return None
    return global_df.iloc[0]


def evaluate_model_row(model_row: pd.Series, zone: str, start_date: str, end_date: str) -> tuple[pd.DataFrame, dict]:
    months = model_row["month_group"] if pd.notna(model_row["month_group"]) else EVAL_MONTHS
    pred_df = prediction_df_for_model(model_row["model_name"], zone, start_date, end_date, months=months)
    pred_df["model_name"] = model_row["model_name"]
    pred_df["month_group_label"] = model_row["month_group_label"]
    metrics = compute_metrics(pred_df["Actual"].to_numpy(), pred_df["Predicted"].to_numpy(), model_row["model_name"])
    metrics["month_group_label"] = model_row["month_group_label"]
    return pred_df, metrics


def stitch_seasonal_predictions(best_models_df: pd.DataFrame, zone: str, start_date: str, end_date: str):
    frames = []
    metric_rows = []
    for _, row in best_models_df.iterrows():
        pred_df, metrics = evaluate_model_row(row, zone, start_date, end_date)
        frames.append(pred_df)
        metric_rows.append(metrics)

    stitched = pd.concat(frames, ignore_index=True).sort_values("Timestamp").reset_index(drop=True)
    if stitched["Timestamp"].duplicated().any():
        dupes = stitched.loc[stitched["Timestamp"].duplicated(), "Timestamp"].head().tolist()
        raise ValueError(f"Overlapping timestamps across seasonal groups: {dupes}")

    overall = compute_metrics(stitched["Actual"].to_numpy(), stitched["Predicted"].to_numpy(), "SeasonalComposite")
    per_group_df = pd.DataFrame(metric_rows).sort_values("MAPE").reset_index(drop=True)
    return stitched, overall, per_group_df


def downsample_for_plot(df: pd.DataFrame, max_points: int = MAX_PLOT_POINTS) -> pd.DataFrame:
    if len(df) <= max_points:
        return df.copy()
    step = max(1, len(df) // max_points)
    return df.iloc[::step].copy()


def plot_predictions(stitched_df: pd.DataFrame, baseline_df: pd.DataFrame | None = None):
    fig = go.Figure()
    stitched_plot = downsample_for_plot(stitched_df)
    fig.add_trace(go.Scatter(x=stitched_plot["Timestamp"], y=stitched_plot["Actual"], mode="lines", name="Actual", line=dict(color="black", width=1.5)))
    fig.add_trace(go.Scatter(x=stitched_plot["Timestamp"], y=stitched_plot["Predicted"], mode="lines", name="Seasonal Composite", line=dict(color="#1f77b4", width=1.5)))

    if baseline_df is not None and not baseline_df.empty:
        baseline_plot = downsample_for_plot(baseline_df)
        fig.add_trace(go.Scatter(x=baseline_plot["Timestamp"], y=baseline_plot["Predicted"], mode="lines", name="Best Global Model", line=dict(color="#d62728", width=1.2, dash="dash")))

    fig.update_layout(
        title=f"{ZONE}: Seasonal Composite vs Actual",
        xaxis_title="Timestamp",
        yaxis_title="Demand",
        template="plotly_white",
        height=600,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    )
    fig.show()


In [7]:
registry = load_registry(REGISTRY_PATH)
df_full = load_data(DATA_PATH)
selection_df = build_selection_table(registry, ZONE)
best_seasonal_models_df = pick_best_models_per_group(selection_df)
best_global_model = pick_best_global_model(selection_df)

print("Best seasonal models by stored month-group test_MAPE:")
display(best_seasonal_models_df[["month_group_label", "model_name", "test_MAPE", "test_MAE", "test_R2"]])

if best_global_model is not None:
    print("Best global baseline model:")
    display(pd.DataFrame([best_global_model[["month_group_label", "model_name", "test_MAPE", "test_MAE", "test_R2"]]]))
else:
    print("No global baseline model with stored test metrics was found.")

Best seasonal models by stored month-group test_MAPE:


,month_group_label,model_name,test_MAPE,test_MAE,test_R2
0,months_1_11_12,nbeats_Total_months_1_11_12_1l_256w_ctx14d_202...,4.410548,213.125302,0.837415
1,months_2_3,nbeats_Total_months_2_3_1l_256w_ctx3d_20260319...,4.629751,222.761097,0.826741
2,months_4_5_6,nbeats_Total_months_4_5_6_1l_256w_ctx7d_202603...,4.459937,215.209505,0.838593
3,months_7_8_9_10,nbeats_Total_months_7_8_9_10_1l_256w_ctx7d_202...,4.438782,214.565884,0.837713


Best global baseline model:


,month_group_label,model_name,test_MAPE,test_MAE,test_R2
0,all_months,nbeats_Total_1l_256w_ctx7d_20260319T160836Z,4.517404,216.684561,0.83446


In [8]:
seasonal_stitched_df, seasonal_overall_metrics, seasonal_group_metrics_df = stitch_seasonal_predictions(
    best_seasonal_models_df,
    zone=ZONE,
    start_date=TEST_START_DATE,
    end_date=TEST_END_DATE,
)

comparison_rows = [seasonal_overall_metrics]
baseline_prediction_df = None

if best_global_model is not None:
    baseline_prediction_df, baseline_metrics = evaluate_model_row(
        best_global_model,
        zone=ZONE,
        start_date=TEST_START_DATE,
        end_date=TEST_END_DATE,
    )
    baseline_metrics["Model"] = "BestGlobalModel"
    comparison_rows.append(baseline_metrics)

comparison_df = pd.DataFrame(comparison_rows).sort_values("MAPE").reset_index(drop=True)

print("Aggregate comparison:")
display(comparison_df[["Model", "MAE", "RMSE", "MAPE", "MAE%", "R2", "TotalAbsError", "Bias", "n_points"]])

print("Per-seasonal-group metrics:")
display(seasonal_group_metrics_df[["month_group_label", "Model", "MAE", "RMSE", "MAPE", "R2", "TotalAbsError", "n_points"]])

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX A4000') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Aggregate comparison:


,Model,MAE,RMSE,MAPE,MAE%,R2,TotalAbsError,Bias,n_points
0,SeasonalComposite,226.952723,302.431862,4.559040,4.517853,0.799803,7.473099e+06,-19.990792,32928
1,BestGlobalModel,230.068705,306.554661,4.665437,4.594036,0.789381,8.172040e+06,8.688708,35520


Per-seasonal-group metrics:


,month_group_label,Model,MAE,RMSE,MAPE,R2,TotalAbsError,n_points
0,months_7_8_9_10,nbeats_Total_months_7_8_9_10_1l_256w_ctx7d_202...,213.343074,273.722809,3.865966,0.585714,2.355308e+06,11040
1,months_2_3,nbeats_Total_months_2_3_1l_256w_ctx3d_20260319...,215.166421,299.550982,4.572784,0.528912,1.136079e+06,5280
2,months_1_11_12,nbeats_Total_months_1_11_12_1l_256w_ctx14d_202...,209.439289,276.880787,4.802365,0.689074,1.809555e+06,8640
3,months_4_5_6,nbeats_Total_months_4_5_6_1l_256w_ctx7d_202603...,272.610137,362.743305,5.246370,0.531166,2.172158e+06,7968


In [9]:
plot_predictions(seasonal_stitched_df, baseline_prediction_df)

seasonal_stitched_df.to_csv(RESULTS_DIR / "seasonal_best_models_eval_predictions.csv", index=False)
seasonal_group_metrics_df.to_csv(RESULTS_DIR / "seasonal_best_models_eval_group_metrics.csv", index=False)
comparison_df.to_csv(RESULTS_DIR / "seasonal_best_models_eval_comparison.csv", index=False)

print("Saved:")
print(RESULTS_DIR / "seasonal_best_models_eval_predictions.csv")
print(RESULTS_DIR / "seasonal_best_models_eval_group_metrics.csv")
print(RESULTS_DIR / "seasonal_best_models_eval_comparison.csv")

Saved:
c:\Users\hp\Desktop\forecasting work\Nbeats refactor\Total\results\seasonal_best_models_eval_predictions.csv
c:\Users\hp\Desktop\forecasting work\Nbeats refactor\Total\results\seasonal_best_models_eval_group_metrics.csv
c:\Users\hp\Desktop\forecasting work\Nbeats refactor\Total\results\seasonal_best_models_eval_comparison.csv
